# Import

In [41]:
import pandas as pd
import numpy as np

# Constantes

In [42]:
# Processed Data Path
PATH_PROCESSED = "../data/processed/"

# Palpites
FILE_TIPS = "palpites_fg_processados.csv"
# Paises
FILE_PAISES = "apoio_paises.csv"

# ETL

## Leitura e Join com apoio

In [43]:
df_paises = pd.read_csv(PATH_PROCESSED + FILE_PAISES)

In [44]:
df_tips_table = pd.read_csv(PATH_PROCESSED + FILE_TIPS)
df_tips_table["nm_pais"] = df_tips_table["nm_time_casa"]

In [45]:
df_tips_table_2 = pd.merge(df_tips_table, df_paises, on='nm_pais', how='left')
df_tips_table_2.drop(["nm_pais"], axis=1, inplace=True)

In [46]:
df_tips_table_3 = df_tips_table_2.dropna()
df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)

C:\Users\ferol\AppData\Local\Temp\ipykernel_23072\4274215766.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_tips_table_3['id_pais'] = df_tips_table_3['id_pais'].astype(int)


## Gerar classificação

In [47]:
df = df_tips_table_3.copy()

# resultados por mandante
home = df.rename(columns={
    "nm_time_casa": "team",
    "nm_time_fora": "opp",
    "vl_time_casa": "gf",
    "vl_time_fora": "ga",
})

In [48]:
home["pts"] = np.select(
    [home["gf"] > home["ga"], home["gf"] == home["ga"]],
    [3, 1],
    default=0,
)

home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [49]:
home["v"] = (home["gf"] > home["ga"]).astype(int)
home["e"] = (home["gf"] == home["ga"]).astype(int)
home["d"] = (home["gf"] < home["ga"]).astype(int)

In [50]:
# resultados por visitante (espelha o jogo)
away = df.rename(columns={
    "nm_time_fora": "team",
    "nm_time_casa": "opp",
    "vl_time_fora": "gf",
    "vl_time_casa": "ga",
})

In [51]:
away["pts"] = np.select(
    [away["gf"] > away["ga"], away["gf"] == away["ga"]],
    [3, 1],
    default=0,
)

away["v"] = (away["gf"] > away["ga"]).astype(int)
away["e"] = (away["gf"] == away["ga"]).astype(int)
away["d"] = (away["gf"] < away["ga"]).astype(int)

In [52]:
# concatena e agrega
team_rows = pd.concat([home, away], ignore_index=True)

In [91]:
base_table = (
    team_rows
    .groupby(["nm_player", "nm_grpo", "team"], as_index=False)
    .agg(
        pts=("pts", "sum"),
        jogos=("team", "size"),
        v=("v", "sum"),
        e=("e", "sum"),
        d=("d", "sum"),
        gp=("gf", "sum"),
        gc=("ga", "sum"),
    )
)

base_table["sg"] = base_table["gp"] - base_table["gc"]

base_table

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5
2,ana nath,A,México,6,3,2,0,1,9,5,4
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4
...,...,...,...,...,...,...,...,...,...,...,...
187,washington,K,Uzbequistão,7,3,2,1,0,7,2,5
188,washington,L,Croácia,3,3,1,0,2,4,5,-1
189,washington,L,Gana,5,3,1,2,0,7,4,3
190,washington,L,Inglaterra,1,3,0,1,2,1,5,-4


In [100]:
base_table['rk'] = base_table.groupby(['nm_player','nm_grpo'])['pts'].rank(method='max', ascending=False).astype(int)

base_table[base_table['nm_player'] == 'ana nath'].sort_values(['nm_grpo', 'rk'], ascending=[True, True])

,nm_player,nm_grpo,team,pts,jogos,v,e,d,gp,gc,sg,rk
1,ana nath,A,Europa D,7,3,2,1,0,7,2,5,1
2,ana nath,A,México,6,3,2,0,1,9,5,4,2
3,ana nath,A,África do Sul,3,3,1,0,2,4,9,-5,3
0,ana nath,A,Coreia do Sul,1,3,0,1,2,3,7,-4,4
4,ana nath,B,Canadá,7,3,2,1,0,7,3,4,1
7,ana nath,B,Suíça,4,3,1,1,1,5,6,-1,2
5,ana nath,B,Catar,3,3,1,0,2,4,6,-2,4
6,ana nath,B,Europa A,3,3,1,0,2,4,5,-1,4
9,ana nath,C,Escócia,7,3,2,1,0,6,2,4,1
10,ana nath,C,Haiti,4,3,1,1,1,7,8,-1,2


In [56]:
jogos = df[
        (df["nm_player"] == base_table["nm_player"].iloc[0]) &
        (df["nm_grpo"] == base_table["nm_grpo"].iloc[0])
    ]

jogos[jogos["result_ref_casa"] == "E"]

tied_teams = set(jogos[jogos["result_ref_casa"] == "E"]["nm_time_casa"])
filt = jogos[
            (jogos["nm_time_casa"].isin(tied_teams)) &
            (jogos["nm_time_fora"].isin(tied_teams))
        ]

jogos[jogos["result_ref_casa"] == "E"]
df




,nm_player,nm_cfr,nm_time_casa,vl_time_casa,nm_time_fora,vl_time_fora,result_ref_casa,id_pais,nm_pais_ajst,nm_grpo
0,ana nath,Alemanha x Costa do Marfim,Alemanha,2,Costa do Marfim,1,V,9,ALEMANHA,E
1,ana nath,Alemanha x Curaçao,Alemanha,4,Curaçao,2,V,9,ALEMANHA,E
2,ana nath,Argentina x Argélia,Argentina,1,Argélia,1,E,3,ARGENTINA,J
3,ana nath,Argentina x Áustria,Argentina,0,Áustria,2,D,3,ARGENTINA,J
4,ana nath,Argélia x Áustria,Argélia,4,Áustria,3,V,28,ARGELIA,J
...,...,...,...,...,...,...,...,...,...,...
283,washington,Uruguai x Cabo Verde,Uruguai,2,Cabo Verde,4,D,15,URUGUAI,H
284,washington,Uruguai x Espanha,Uruguai,2,Espanha,1,V,15,URUGUAI,H
285,washington,Uzbequistão x Colômbia,Uzbequistão,2,Colômbia,2,E,33,UZBEQUISTAO,K
286,washington,África do Sul x Coreia do Sul,África do Sul,4,Coreia do Sul,0,V,36,AFRICA DO SUL,A


## NAO FUNCIONANDO

In [19]:
def sort_group(part_df):
    # part_df: linhas de um participante+grupo, já com stats agregados
    jogos = df[
        (df["nm_player"] == part_df["nm_player"].iloc[0]) &
        (df["nm_grpo"] == part_df["nm_grpo"].iloc[0])
    ]
    # helper para head-to-head entre subconjunto de times empatados
    def head_to_head(tied_teams):
        filt = jogos[
            (jogos["nm_time_casa"].isin(tied_teams)) &
            (jogos["nm_time_fora"].isin(tied_teams))
        ]
        if filt.empty:
            return None
        h = filt.rename(columns={"nm_time_casa":"team","nm_time_fora":"opp","vl_casa":"gf","vl_fora":"ga"})
        h["pts"] = np.select([h["gf"]>h["ga"], h["gf"]==h["ga"]],[3,1],default=0)
        h["sg"] = h["gf"]-h["ga"]
        a = filt.rename(columns={"nm_time_fora":"team","nm_time_casa":"opp","vl_fora":"gf","vl_casa":"ga"})
        a["pts"] = np.select([a["gf"]>a["ga"], a["gf"]==a["ga"]],[3,1],default=0)
        a["sg"] = a["gf"]-a["ga"]
        hh = pd.concat([h,a])
        return (
            hh.groupby("team", as_index=False)
              .agg(pts=("pts","sum"), sg=("sg","sum"), gp=("gf","sum"))
        )

    # ordena com desempates iterativos
    part_df = part_df.copy()
    part_df["order"] = 0  # será sobrescrita
    # passo base: pontos gerais
    part_df = part_df.sort_values(["pts","sg","gp","team"], ascending=[False,False,False,True])

    i = 0
    while i < len(part_df):
        # encontra bloco empatado em pontos
        same_pts = part_df.iloc[i]["pts"]
        block = part_df.index[part_df["pts"] == same_pts].tolist()
        j = i
        while j < len(part_df) and part_df.iloc[j]["pts"] == same_pts:
            j += 1
        block_idx = part_df.index[i:j]

        # aplica head-to-head se mais de 1 time empatado
        if len(block_idx) > 1:
            tied_teams = part_df.loc[block_idx, "team"].tolist()
            hh = head_to_head(tied_teams)
            if hh is not None:
                # mescla stats de confronto direto
                merged = part_df.loc[block_idx].merge(hh, on="team", how="left", suffixes=("", "_hh"))
                merged = merged.sort_values(
                    ["pts_hh","sg_hh","gp_hh","sg","gp","team"],
                    ascending=[False,False,False,False,False,True]
                )
                part_df.loc[merged.index, "order"] = range(i+1, i+1+len(merged))
                part_df = pd.concat([part_df.drop(index=merged.index), merged]).sort_values("order").drop(columns="order")
            i = j
        else:
            i += 1

    # se ainda restarem empates exatos, fallback no sort final:
    return part_df.sort_values(["pts","sg","gp","team"], ascending=[False,False,False,True])


In [20]:
standings = (
    base_table
    .groupby(["nm_player", "nm_grpo"], group_keys=False)
    .apply(sort_group)
    .reset_index(drop=True)
)

# opcional: numerar posições por grupo
standings["pos"] = (
    standings.groupby(["nm_player","nm_grpo"])
             .cumcount() + 1
)

print(standings.head())


KeyError: 'gf'